# Topic Graph (preview): Answering "What Are the Main Themes?"

A **topic graph** groups related chunks into **topics** (also called communities), arranged in a hierarchy. At query time, **topic-aware retrieval samples evidence across topics** instead of returning only the closest vector matches, so broad questions get answers that cover the whole collection.

| | Topic graph | Entity graph ([02-entity-graph](02-entity-graph.ipynb)) |
|---|---|---|
| Represents | Themes and their hierarchy | Typed entities and their relationships |
| Good for | "What are the main themes…?", "What topics does this collection cover?" | "Which astronauts launched from sites in Kazakhstan?" |
| Ontology | Not needed | Required |
| Built by | **Client-side**: the Foundry IQ skills plugin | **Server-side**: a graph-enabled Blob knowledge source |

> **There is no server-side topic graph builder in Foundry IQ (as of November 2026).** The service *uses* a prepared topic index at query time (`topic_graph`: none, balanced or diverse), but nothing in a skillset or knowledge source *creates* one. The supported way to build it is the `topic-graph-build` command of the [Foundry IQ skills plugin](../docs/foundry-iq-skills-guide.md), which runs a native builder on your machine and uploads the result.

```mermaid
flowchart LR
    IDX[("space-exploration<br/>chunk index")] -->|chunk vectors| Build
    subgraph Build["Foundry IQ skills plugin (client-side)"]
        direction TB
        S["Similarity graph<br/>chunk ↔ nearest chunks"] --> C["Community detection<br/>→ topic hierarchy"]
        C --> CEN["Centroid vector per topic"]
        CEN --> N["LLM names each topic"]
    end
    Build --> TI[("Topic index")]
    TI -. "topic_graph_profile on text_vector" .-> IDX
    IDX --> KS["searchIndex knowledge source<br/>topic_graph: none | balanced | diverse"]
    KS --> KB["Knowledge base"]

    style TI fill:#f0f9ff,stroke:#0078d4,color:#000
    style Build fill:#fffdf0,stroke:#d4a000,color:#000
```

## Prerequisites

| Requirement | Notes |
|---|---|
| [00-setup/prepare-space-dataset.ipynb](../00-setup/prepare-space-dataset.ipynb) completed | `text_vector` is stored and retrievable with a cosine HNSW profile, as the builder requires |
| Search service with topic graph support in its region | |
| Roles: **Search Service Contributor**, **Search Index Data Contributor**, **Search Index Data Reader** | The plugin requires Microsoft Entra sign-in (`az login`); it doesn't accept API keys |
| GitHub Copilot CLI, Git, Node.js, Python 3.12–3.14, Azure CLI, **Windows x64 or Linux** | The native topic builder doesn't run on macOS |
| A chat deployment for naming topics | Chosen during the plugin's planning step |

## Setup

In [ ]:
# %pip install azure-search-documents==12.2.0b1 azure-identity python-dotenv

In [ ]:
import os
from collections import Counter

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
    KnowledgeSourceResultsProcessing,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalSemanticIntent,
    SearchIndexKnowledgeSourceParams,
)
from azure.search.documents.models import (
    BalancedVectorQueryTopicGraph,
    DiverseVectorQueryTopicGraph,
    NoneVectorQueryTopicGraph,
    VectorizableTextQuery,
)
from dotenv import load_dotenv

load_dotenv(override=True)  # pick up .env edits without restarting the kernel

search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
search_api_key = os.environ.get("AZURE_SEARCH_API_KEY")
search_credential = AzureKeyCredential(search_api_key) if search_api_key else DefaultAzureCredential()

CONTENT_INDEX = "space-exploration"
VECTOR_FIELD = "text_vector"
KNOWLEDGE_SOURCE = "space-topic-ks"
KNOWLEDGE_BASE = "space-topic-kb"
BROAD_QUESTION = "What are the main themes and eras of space exploration covered in these documents?"

index_client = SearchIndexClient(search_endpoint, search_credential)
content_client = SearchClient(search_endpoint, CONTENT_INDEX, search_credential)

print(f"Search endpoint: {search_endpoint}")

## Step 1: Why a topic graph?

Vector search returns the *k* chunks **closest** to the question. For a broad question the closest chunks tend to sit in the same corner of the embedding space, for example several overview paragraphs about the Space Race, so the evidence is **redundant** and misses other themes (Mars rovers, space telescopes, commercial launchers…).

Let's measure that with a plain vector query: how many of the top 20 chunks come from distinct articles?

In [ ]:
plain = content_client.search(
    vector_queries=[VectorizableTextQuery(text=BROAD_QUESTION, k_nearest_neighbors=20, fields=VECTOR_FIELD)],
    select=["chunk_id", "title"],
    top=20,
)
plain_titles = Counter(doc["title"] for doc in plain)
print(f"Top 20 chunks come from {len(plain_titles)} distinct articles:")
for title, n in plain_titles.most_common():
    print(f"  {n:>2} × {title}")

## Step 2: How a topic graph is built (the concept)

The plugin's native builder follows four steps:

1. **Similarity graph:** each chunk is a node, connected to its nearest neighbours by vector similarity
2. **Community detection:** densely connected groups of chunks become **topics**. Running detection at several resolutions gives a **hierarchy** (broad themes → narrow leaf topics, at most 100 chunks each)
3. **Centroids:** each topic gets a representative vector, the normalised mean of its chunks' vectors
4. **Naming:** an LLM reads a few chunks closest to each centroid and gives the topic a short label

The **topic index** stores one document per topic (identifier, label, centroid vector, hierarchy). The content index's vector field is linked to it through a **`topic_graph_profile`**. At query time, retrieval gathers a candidate pool (`min_diversity_pool_size`, 250–10,000) and **samples across topics** using the centroids, governed by:

| `topic_graph` | Behaviour |
|---|---|
| `NoneVectorQueryTopicGraph` | Plain vector search |
| `BalancedVectorQueryTopicGraph` | Recommended default; blends similarity with diversity (fixed diversity factor 0.2) |
| `DiverseVectorQueryTopicGraph` | Stronger diversity; tune `diversity_factor` (0–1) |

## Step 3: Create the knowledge source and knowledge base

Create a `searchIndex` knowledge source over the content index, searching the **vector field**. Don't set a stored `topic_graph` default yet: the service rejects it until the vector field is linked to a topic index (Step 4). Later we set `topic_graph` per request. The plugin's retrieval setup step (Step 4) expects this knowledge source and knowledge base to already exist.

In [ ]:
# No topic_graph default here: the service rejects it until text_vector has a topicGraphProfile (Step 4).
knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE,
    results_processing=KnowledgeSourceResultsProcessing.NONE,
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=CONTENT_INDEX,
        search_fields=[SearchIndexFieldReference(name=VECTOR_FIELD)],
        source_data_fields=[SearchIndexFieldReference(name=f) for f in ("chunk_id", "title", "chunk")],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE,
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE)],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)
index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge source '{KNOWLEDGE_SOURCE}' and knowledge base '{KNOWLEDGE_BASE}' ready")

## Step 4: Build the topic graph with the Foundry IQ skills plugin

Run these steps in **GitHub Copilot CLI**, not in this notebook. See [docs/foundry-iq-skills-guide.md](../docs/foundry-iq-skills-guide.md) for full detail.

**1. Install the plugin** (slash commands inside Copilot CLI):

```text
/plugin marketplace add microsoft/azure-skills
/plugin install foundry-iq-skills@azure-skills
```

**2. Sign in** with Microsoft Entra (the plugin doesn't use API keys):

```bash
az login --tenant <tenant-id-or-domain>
```

**3. Ask Copilot to plan the build.** Give it your names explicitly; the skill never guesses resources:

> *Check whether this machine is ready to build a Topic Graph. Then inspect the source index `space-exploration` on search service `<search-service>` (resource group `<rg>`, subscription `<sub>`) and plan a Topic Graph build. Use key field `chunk_id`, text field `chunk`, source field `source_url`, title field `title`, vector field `text_vector`, embedding model `text-embedding-3-large`, and name topics with chat deployment `<deployment>` on `<foundry-endpoint>`.*

Behind the scenes the skill runs `topic-graph-build … doctor`, `inspect` and `plan`, then shows an **approval card** (headline, time and cost, maximum model calls). Approve it.

**4. Apply and verify.** The skill runs `apply` (the native TopicGraph executable reads your chunk vectors, builds the hierarchy, names topics, and uploads a **new topic index**) and then `verify`.

**5. Turn on topic retrieval** (a separate approval):

> *Turn on topic retrieval for the verified generation using knowledge source `space-topic-ks` and knowledge base `space-topic-kb`. Keep the stored default off.*

This runs `plan-retrieval` → `configure-retrieval` → `verify-retrieval`, which links `text_vector` to the topic index through a `topic_graph_profile`.

When that's done, come back here and continue with Step 5.

## Step 5: Inspect what the plugin created

The topic index schema is **produced by the plugin** and isn't documented elsewhere, so reading it back is the best way to see what a topic graph actually contains.

### Inspection helpers

In [ ]:
def linked_topic_index() -> tuple[str | None, str | None]:
    """Return (profile name, topic index name) linked to the content index's vector field."""
    content_index = index_client.get_index(CONTENT_INDEX)
    vector_field = next(f for f in content_index.fields if f.name == VECTOR_FIELD)
    profiles = content_index.vector_search.topic_graph_profiles or []
    profile = next((p for p in profiles if p.name == vector_field.topic_graph_profile), None)
    return vector_field.topic_graph_profile, (profile.index_name if profile else None)


def describe_topic_index(name: str, sample: int = 5) -> None:
    topic_index = index_client.get_index(name)
    print(f"Topic index '{name}' fields:")
    for field in topic_index.fields:
        flags = [k for k in ("key", "searchable", "filterable", "retrievable", "stored") if getattr(field, k, None)]
        print(f"  {field.name:<28} {field.type:<26} dims={field.vector_search_dimensions or ''} {flags}")
    results = SearchClient(search_endpoint, name, search_credential).search(search_text="*", include_total_count=True, top=sample)
    rows = list(results)
    print(f"\n{results.get_count()} topic documents. Sample (vectors truncated):")
    for doc in rows:
        print({k: (v[:3] + ["…"] if isinstance(v, list) and len(v) > 3 else v) for k, v in doc.items() if not k.startswith("@")})

### Run it

In [ ]:
PROFILE_NAME, TOPIC_INDEX = linked_topic_index()
print(f"{VECTOR_FIELD}.topic_graph_profile = {PROFILE_NAME}")
print(f"Linked topic index: {TOPIC_INDEX or 'none yet — complete Step 4 first'}")

In [ ]:
if TOPIC_INDEX:
    describe_topic_index(TOPIC_INDEX)

## Step 6: Compare `none`, `balanced` and `diverse`

Send the same broad question three times, overriding `topic_graph` per request, and measure how many **distinct articles** the evidence covers.

### Comparison helpers

In [ ]:
kb_client = KnowledgeBaseRetrievalClient(search_endpoint, search_credential, knowledge_base_name=KNOWLEDGE_BASE)

TOPIC_SETTINGS = {
    "none": NoneVectorQueryTopicGraph(),
    "balanced": BalancedVectorQueryTopicGraph(min_diversity_pool_size=1000),
    "diverse (0.5)": DiverseVectorQueryTopicGraph(diversity_factor=0.5, min_diversity_pool_size=1000),
}


def retrieve_with_topics(topic_graph) -> list[dict]:
    request = KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=BROAD_QUESTION)],
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        include_activity=True,
        knowledge_source_params=[SearchIndexKnowledgeSourceParams(
            knowledge_source_name=KNOWLEDGE_SOURCE,
            include_references=True,
            include_reference_source_data=True,
            topic_graph=topic_graph,
        )],
    )
    return [dict(ref.source_data or {}) for ref in kb_client.retrieve(request).references or []]


def compare_topic_settings() -> None:
    for label, setting in TOPIC_SETTINGS.items():
        refs = retrieve_with_topics(setting)
        titles = Counter(r.get("title") for r in refs)
        print(f"{label:<14} {len(refs):>2} chunks from {len(titles):>2} articles: {[t for t, _ in titles.most_common(6)]}")

### Run it

In [ ]:
if TOPIC_INDEX:
    compare_topic_settings()
else:
    print("No topic index linked yet; complete Step 4 first.")

With `balanced` and `diverse`, evidence should spread across more articles and themes (early Space Race, lunar programmes, space stations, Mars, telescopes, commercial spaceflight) instead of concentrating on one.

## Clean up

Delete the knowledge base and knowledge source created here. Retire the topic index through the plugin (*"retire my topic graph staging"*) so its own state stays consistent; deleting it by hand would leave the content index's `topic_graph_profile` pointing at a missing index.

In [ ]:
# index_client.delete_knowledge_base(KNOWLEDGE_BASE)
# index_client.delete_knowledge_source(KNOWLEDGE_SOURCE)
# print("Deleted knowledge base and knowledge source")

## Summary

- **Why:** plain vector search returns near-duplicate evidence for broad questions; topic sampling spreads evidence across themes
- **How:** similarity graph → communities (hierarchy) → centroids → LLM labels → topic index linked through `topic_graph_profile`
- **Who builds it:** the client, using the Foundry IQ skills plugin's `topic-graph-build`
- **Query time:** `topic_graph` on the knowledge source (stored default) or per request: none, balanced (recommended) or diverse

### Next steps

- [02-entity-graph.ipynb](02-entity-graph.ipynb): relationship questions with an entity graph
- [docs/foundry-iq-skills-guide.md](../docs/foundry-iq-skills-guide.md): the full plugin workflow